In [ ]:
!pip install pymupdf opencv-python pillow matplotlib pandas

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import fitz

pdf_path = list(uploaded.keys())[0]

doc = fitz.open(pdf_path)

page_output_dir = "outputs/pages"
os.makedirs(page_output_dir, exist_ok=True)

for page_index in range(len(doc)):
    page = doc[page_index]

    matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=matrix)

    page_number = page_index + 1
    image_path = f"{page_output_dir}/page_{page_number:03}.png"

    pix.save(image_path)

    print(f"{page_number}/{len(doc)} 페이지 이미지 저장 완료")

print("전체 페이지 이미지 변환 완료")
print("전체 페이지 수:", len(doc))

In [ ]:
page_files = sorted([
    f for f in os.listdir(page_output_dir)
    if f.endswith(".png")
])

print("생성된 페이지 이미지 개수:", len(page_files))
print("처음 5개:", page_files[:5])
print("마지막 5개:", page_files[-5:])

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-kor
!apt-get install -y tesseract-ocr-eng
!pip install pytesseract

In [ ]:
import cv2
import json
import numpy as np

def detect_blocks(image_path):
    image = cv2.imread(image_path)
    page_height, page_width = image.shape[:2]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 글자와 선을 강조하기 위한 이진화
    _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    # 글자와 선을 블록 단위로 묶기
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 10))
    dilated = cv2.dilate(binary, kernel, iterations=2)

    contours, _ = cv2.findContours(
        dilated,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    blocks = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)

        # 너무 작은 잡음 제거
        if w > 50 and h > 30:
            blocks.append({
                "bbox": [int(x), int(y), int(x + w), int(y + h)],
                "width": int(w),
                "height": int(h)
            })

    # 위에서 아래, 왼쪽에서 오른쪽 순서로 정렬
    blocks = sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))

    return blocks, page_width, page_height

In [ ]:
def is_inside(inner_bbox, outer_bbox):
    ix1, iy1, ix2, iy2 = inner_bbox
    ox1, oy1, ox2, oy2 = outer_bbox

    return ix1 >= ox1 and iy1 >= oy1 and ix2 <= ox2 and iy2 <= oy2


def remove_duplicate_blocks(blocks):
    filtered_blocks = []

    for i, block in enumerate(blocks):
        bbox = block["bbox"]
        inside_other = False

        for j, other_block in enumerate(blocks):
            if i == j:
                continue

            other_bbox = other_block["bbox"]

            # 현재 박스가 다른 큰 박스 안에 거의 포함되면 제거
            if is_inside(bbox, other_bbox):
                current_area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
                other_area = (other_bbox[2] - other_bbox[0]) * (other_bbox[3] - other_bbox[1])

                if current_area < other_area * 0.8:
                    inside_other = True
                    break

        if not inside_other:
            filtered_blocks.append(block)

    return filtered_blocks

In [ ]:
import cv2
import re
import pytesseract
from PIL import Image

def clean_block_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_text_from_bbox(image, bbox, pad=5):
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]

    x1 = max(x1 - pad, 0)
    y1 = max(y1 - pad, 0)
    x2 = min(x2 + pad, w)
    y2 = min(y2 + pad, h)

    crop = image[y1:y2, x1:x2]

    if crop.size == 0:
        return ""

    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(crop_rgb)

    text = pytesseract.image_to_string(
        pil_img,
        lang="kor+eng",
        config="--psm 6"
    )

    return clean_block_text(text)


def is_table_like(crop):
    if crop is None or crop.size == 0:
        return False

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(
        gray,
        180,
        255,
        cv2.THRESH_BINARY_INV
    )

    h, w = binary.shape[:2]

    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (max(w // 15, 30), 1)
    )

    vertical_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (1, max(h // 15, 30))
    )

    horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)
    vertical_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel)

    horizontal_score = cv2.countNonZero(horizontal_lines) / (h * w)
    vertical_score = cv2.countNonZero(vertical_lines) / (h * w)

    line_score = horizontal_score + vertical_score

    return line_score > 0.015


def classify_block(block, page_width, page_height, image=None):
    x1, y1, x2, y2 = block["bbox"]
    w = x2 - x1
    h = y2 - y1

    width_ratio = w / page_width
    height_ratio = h / page_height
    y_ratio = y1 / page_height

    text = block.get("ocr_text", "")
    compact_text = text.replace(" ", "").replace("\n", "")

    crop = None
    if image is not None:
        crop = image[y1:y2, x1:x2]

    # 1. 하단 페이지 번호/단원명
    if y_ratio > 0.9:
        return "footer"

    # 2. 표 영역
    # 표는 선이 많으므로 우선적으로 table로 분류
    if width_ratio > 0.55 and height_ratio > 0.12:
        if is_table_like(crop):
            return "table"

    # 3. 예제 + 풀이 박스
    # 예제, 풀이 키워드가 OCR로 잡히는 경우
    if ("예제" in compact_text or "보기" in compact_text or "문제" in compact_text) and ("풀이" in compact_text or "해설" in compact_text):
        return "example_solution_box"

    # 4. 중간 위치의 큰 설명 박스
    # 교과서 예제 박스는 보통 페이지 중간에 넓고 크게 배치됨
    # 그래프는 보통 페이지 하단에 있으므로 y_ratio로 구분
    if width_ratio > 0.55 and height_ratio > 0.10 and 0.20 < y_ratio < 0.58:
        return "example_solution_box"

    # 5. 참고 설명
    if "참고" in compact_text:
        return "reference_text"

    # 6. 그래프/그림 영역
    # 하단에 위치한 넓은 시각자료 영역
    if width_ratio > 0.55 and height_ratio > 0.12 and y_ratio >= 0.58:
        return "graph_or_figure"

    # 7. 표 제목/그래프 범례/캡션
    if width_ratio < 0.4 and height_ratio < 0.06 and y_ratio > 0.35:
        return "caption_or_legend"

    if 0.25 < width_ratio < 0.65 and height_ratio < 0.06 and y_ratio > 0.5:
        return "caption_or_legend"

    # 8. 공식/수식 박스
    center_x = (x1 + x2) / 2
    page_center_x = page_width / 2

    formula_keywords = ["=", "+", "-", "×", "/", "GDP", "100", "%"]

    if abs(center_x - page_center_x) < page_width * 0.28:
        if 0.25 < width_ratio < 0.75 and height_ratio < 0.16:
            if any(keyword in compact_text for keyword in formula_keywords):
                return "formula_box"

    # 9. 상단 제목/헤더
    if y_ratio < 0.15 and height_ratio < 0.09 and 0.45 < width_ratio < 0.75:
        return "title_or_header"

    # 10. 작은 요소
    if width_ratio < 0.15 and height_ratio < 0.08:
        return "small_element"

    # 11. 나머지는 일반 본문
    return "body_text"

In [ ]:
import os
import json

layout_output_dir = "outputs/layout_results"
os.makedirs(layout_output_dir, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(page_output_dir)
    if f.endswith(".png")
])

print("구조화 대상 페이지 수:", len(image_files))

for idx, file_name in enumerate(image_files):
    image_path = os.path.join(page_output_dir, file_name)
    page_number = int(file_name.replace("page_", "").replace(".png", ""))

    image = cv2.imread(image_path)

    blocks, page_width, page_height = detect_blocks(image_path)
    blocks = remove_duplicate_blocks(blocks)

    structured_blocks = []

    for block_idx, block in enumerate(blocks):
        block_id = f"p{page_number:03}_b{block_idx + 1:03}"

        ocr_text = extract_text_from_bbox(image, block["bbox"])

        block["ocr_text"] = ocr_text

        block_type = classify_block(
            block,
            page_width,
            page_height,
            image=image
        )

        structured_blocks.append({
            "block_id": block_id,
            "type": block_type,
            "bbox": block["bbox"],
            "ocr_text": ocr_text
        })

    page_result = {
        "page_id": page_number,
        "image_path": image_path,
        "page_width": page_width,
        "page_height": page_height,
        "block_count": len(structured_blocks),
        "blocks": structured_blocks
    }

    save_path = f"{layout_output_dir}/page_{page_number:03}_layout.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(page_result, f, ensure_ascii=False, indent=2)

    print(f"{idx + 1}/{len(image_files)} 구조화 완료: {save_path}")

print("전체 페이지 세분화 구조화 완료")

In [ ]:
layout_files = sorted([
    f for f in os.listdir(layout_output_dir)
    if f.endswith(".json")
])

print("구조화 JSON 파일 개수:", len(layout_files))
print("처음 5개:", layout_files[:5])
print("마지막 5개:", layout_files[-5:])

In [ ]:
import pandas as pd

layout_files = sorted([
    f for f in os.listdir(layout_output_dir)
    if f.endswith(".json")
])

summary_data = []

for file_name in layout_files:
    json_path = os.path.join(layout_output_dir, file_name)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    type_counts = {}

    for block in data["blocks"]:
        block_type = block["type"]
        type_counts[block_type] = type_counts.get(block_type, 0) + 1

    summary_data.append({
        "page_id": data["page_id"],
        "block_count": data["block_count"],
        "title_or_header": type_counts.get("title_or_header", 0),
        "body_text": type_counts.get("body_text", 0),
        "formula_box": type_counts.get("formula_box", 0),
        "example_solution_box": type_counts.get("example_solution_box", 0),
        "reference_text": type_counts.get("reference_text", 0),
        "caption_or_legend": type_counts.get("caption_or_legend", 0),
        "table": type_counts.get("table", 0),
        "graph_or_figure": type_counts.get("graph_or_figure", 0),
        "footer": type_counts.get("footer", 0),
        "small_element": type_counts.get("small_element", 0)
    })

summary_df = pd.DataFrame(summary_data)

summary_path = "outputs/layout_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("세분화 구조화 요약 CSV 저장 완료:", summary_path)
summary_df.head(10)

In [ ]:
sample_json_path = "outputs/layout_results/page_008_layout.json"

with open(sample_json_path, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(json.dumps(sample_data, ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt

sample_page = 8
sample_image_path = f"outputs/pages/page_{sample_page:03}.png"
sample_json_path = f"outputs/layout_results/page_{sample_page:03}_layout.json"

image = cv2.imread(sample_image_path)

with open(sample_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

for block in data["blocks"]:
    x1, y1, x2, y2 = block["bbox"]
    block_id = block["block_id"]

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)
    cv2.putText(
        image,
        block_id,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

layout_sample_dir = "outputs/layout_sample"
os.makedirs(layout_sample_dir, exist_ok=True)

save_image_path = f"{layout_sample_dir}/page_{sample_page:03}_layout_visualization.png"
cv2.imwrite(save_image_path, image)

print("시각화 이미지 저장 완료:", save_image_path)

plt.figure(figsize=(10, 14))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import shutil

layout_sample_dir = "outputs/layout_sample"
os.makedirs(layout_sample_dir, exist_ok=True)

sample_pages = [8, 16]

for page_number in sample_pages:
    src = f"outputs/layout_results/page_{page_number:03}_layout.json"
    dst = f"{layout_sample_dir}/page_{page_number:03}_layout.json"

    shutil.copy(src, dst)

    print("샘플 JSON 복사 완료:", dst)

In [ ]:
import os
import json
import pandas as pd

evaluation_dir = "outputs/evaluation"
os.makedirs(evaluation_dir, exist_ok=True)

layout_output_dir = "outputs/layout_results"
summary_path = "outputs/layout_summary.csv"

print("Evaluation 결과 저장 폴더:", evaluation_dir)

In [ ]:
layout_files = sorted([
    f for f in os.listdir(layout_output_dir)
    if f.endswith(".json")
])

print("구조화 JSON 파일 개수:", len(layout_files))
print("처음 5개:", layout_files[:5])
print("마지막 5개:", layout_files[-5:])

In [ ]:
evaluation_rows = []

for file_name in layout_files:
    json_path = os.path.join(layout_output_dir, file_name)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    block_count = data.get("block_count", len(data.get("blocks", [])))

    type_counts = {}

    for block in data["blocks"]:
        block_type = block["type"]
        type_counts[block_type] = type_counts.get(block_type, 0) + 1

    # 간단 평가 기준
    if block_count == 0:
        status = "error_no_block"
    elif block_count <= 2:
        status = "check_too_few_blocks"
    elif block_count >= 25:
        status = "check_too_many_blocks"
    else:
        status = "normal"

    evaluation_rows.append({
        "page_id": data["page_id"],
        "block_count": block_count,

        "title_or_header": type_counts.get("title_or_header", 0),
        "body_text": type_counts.get("body_text", 0),
        "formula_box": type_counts.get("formula_box", 0),
        "example_solution_box": type_counts.get("example_solution_box", 0),
        "reference_text": type_counts.get("reference_text", 0),
        "caption_or_legend": type_counts.get("caption_or_legend", 0),
        "table": type_counts.get("table", 0),
        "graph_or_figure": type_counts.get("graph_or_figure", 0),
        "footer": type_counts.get("footer", 0),
        "small_element": type_counts.get("small_element", 0),

        "status": status
    })

layout_eval_df = pd.DataFrame(evaluation_rows)

layout_eval_path = "outputs/evaluation/layout_evaluation_summary.csv"
layout_eval_df.to_csv(layout_eval_path, index=False, encoding="utf-8-sig")

print("저장 완료:", layout_eval_path)
layout_eval_df.head(10)

In [ ]:
print("전체 페이지 수:", len(layout_eval_df))
print("평균 블록 수:", round(layout_eval_df["block_count"].mean(), 2))
print("최소 블록 수:", layout_eval_df["block_count"].min())
print("최대 블록 수:", layout_eval_df["block_count"].max())

print("\n상태별 페이지 수:")
print(layout_eval_df["status"].value_counts())

In [ ]:
sample_evaluation = [
    {
        "page_id": 8,
        "expected_elements": "formula_box, body_text, caption_or_legend, table, footer",
        "detected_result": "상단 수식 박스는 formula_box로, 본문은 body_text로, 표 제목은 caption_or_legend로, 표 영역은 table로 분리됨",
        "is_success": "Y",
        "issue": "표 영역은 감지되었으나 행/열 단위 세부 구조 복원은 추가 표 처리 필요"
    },
    {
        "page_id": 16,
        "expected_elements": "body_text, formula_box, example_solution_box, reference_text, graph_or_figure, caption_or_legend",
        "detected_result": "본문, 공식 박스, 예제/풀이 박스, 참고 설명, 그래프 영역, 그래프 범례가 각각 분리됨",
        "is_success": "Y",
        "issue": "그래프는 graph_or_figure로 감지되었으나 그래프 내부의 축, 범례, 수치 해석은 Model B 단계와 연계 필요"
    }
]

sample_eval_df = pd.DataFrame(sample_evaluation)

sample_eval_path = "outputs/evaluation/sample_page_evaluation.csv"
sample_eval_df.to_csv(sample_eval_path, index=False, encoding="utf-8-sig")

print("저장 완료:", sample_eval_path)
sample_eval_df

In [ ]:
import cv2
import json
import os
import matplotlib.pyplot as plt

sample_page = 16

sample_image_path = f"outputs/pages/page_{sample_page:03}.png"
sample_json_path = f"outputs/layout_results/page_{sample_page:03}_layout.json"

image = cv2.imread(sample_image_path)

with open(sample_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

for block in data["blocks"]:
    x1, y1, x2, y2 = block["bbox"]
    block_id = block["block_id"]
    block_type = block["type"]

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)

    label = f"{block_id} {block_type}"

    cv2.putText(
        image,
        label,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2
    )

layout_sample_dir = "outputs/layout_sample"
os.makedirs(layout_sample_dir, exist_ok=True)

save_image_path = f"{layout_sample_dir}/page_{sample_page:03}_layout_visualization.png"
cv2.imwrite(save_image_path, image)

print("16페이지 시각화 이미지 저장 완료:", save_image_path)

plt.figure(figsize=(10, 14))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
total_pages = len(layout_eval_df)
avg_blocks = round(layout_eval_df["block_count"].mean(), 2)
min_blocks = layout_eval_df["block_count"].min()
max_blocks = layout_eval_df["block_count"].max()
status_counts = layout_eval_df["status"].value_counts().to_dict()

report = f"""# Model A Evaluation Report

## 1. Evaluation 개요

본 문서는 Model A의 문서 구조화 및 전처리 결과를 평가하기 위한 보고서이다.

Model A에서는 PDF 페이지 분할, OCR 및 텍스트 정제, 문서 구조화, 블록 단위 좌표 생성 과정을 수행하였다. 본 Evaluation 단계에서는 전체 페이지에 대해 구조화 결과가 정상적으로 생성되었는지, 블록 감지 결과가 적절한지, 샘플 페이지 기준으로 예상 문서 요소가 잘 분리되었는지를 확인하였다.

## 2. 구조화 결과 생성 여부

- 전체 대상 페이지 수: 198페이지
- 생성된 layout JSON 파일 수: {total_pages}개
- 평균 블록 수: {avg_blocks}
- 최소 블록 수: {min_blocks}
- 최대 블록 수: {max_blocks}

## 3. 상태별 페이지 수

{status_counts}

## 4. 세분화된 블록 유형

본 프로젝트에서는 단순히 본문과 표만 구분하는 것이 아니라, 시각장애인 접근성 변환에 필요한 문서 요소를 더 세분화하였다.

- body_text: 일반 본문
- formula_box: 공식 또는 수식 박스
- example_solution_box: 예제 및 풀이 박스
- reference_text: 참고 설명
- caption_or_legend: 표 제목 또는 그래프 범례
- table: 표 영역
- graph_or_figure: 그래프 또는 그림 영역
- footer: 하단 페이지 정보
- small_element: 작은 장식 또는 보조 요소

## 5. 샘플 페이지 평가

8페이지의 경우 수식 박스, 본문, 표 제목, 표 영역, footer 영역이 각각 분리되었다. 특히 표 영역은 table 블록으로 감지되어 일반 OCR에서 표 구조가 깨지는 문제를 보완할 수 있는 기반을 마련하였다.

16페이지의 경우 일반 본문, 공식 박스, 예제/풀이 박스, 참고 설명, 그래프 영역, 그래프 범례가 분리되었다. 이를 통해 텍스트 중심 영역과 시각자료 중심 영역을 구분할 수 있음을 확인하였다.

## 6. 확인된 한계

- OpenCV 기반 구조화 방식은 규칙 기반이므로 모든 페이지에서 완벽한 분류를 보장하지 않는다.
- OCR 결과에 따라 예제, 참고, 풀이 등의 키워드 인식이 불완전할 수 있다.
- 표 영역은 하나의 table 블록으로 감지되지만, 행과 열 단위의 세부 구조 복원은 추가 처리가 필요하다.
- 그래프 영역은 graph_or_figure로 감지되지만, 그래프 내부의 축, 범례, 수치 해석은 Model B 단계와 연계해야 한다.

## 7. 개선 방향

- 예제, 풀이, 참고, 공식 등 교과서 고유 문서 요소에 대한 라벨링 기준을 더 정교화한다.
- 표 내부의 행/열 구조를 별도로 추출하는 후처리 로직을 추가한다.
- 그래프 영역은 Model B의 시각자료 해석 모델과 연계하여 대체 텍스트를 생성한다.
- 향후 DocLayout-YOLO, LayoutParser 등 문서 레이아웃 분석 모델과 비교 실험을 진행할 수 있다.

## 8. 결론

Model A는 전체 198페이지에 대해 페이지 단위 이미지 변환, OCR, 문서 구조화, 블록 ID 및 bbox 좌표 생성을 수행하였다. Evaluation 결과, 전체 페이지에 대한 구조화 JSON 생성이 완료되었으며, 대표 샘플 페이지에서 본문, 수식, 예제/풀이, 표, 그래프, 범례 영역이 세분화되어 분리되는 것을 확인하였다.

다만 표 세부 구조 복원과 그래프 내부 의미 해석은 추가 모델과의 연계가 필요하다.
"""

report_path = "outputs/evaluation/modelA_evaluation_report.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("보고서 저장 완료:", report_path)